# 44｜从零实现 Mamba 风格 Selective SSM：逐步 Scan、门控与因果卷积

本 Notebook 用基础 PyTorch 写一个便于审计的 Mamba 风格教学模型：输入相关的 delta、B、C，稳定的负 A 参数化，逐时间步 selective scan，depthwise causal convolution，SiLU gate，残差 block 与自回归 LM forward。

这里明确不是官方 Mamba kernel，也不声称复现吞吐：Python for-loop 的 scan 是为了让状态更新逐行可见。官方实现依赖融合 CUDA/Triton kernel、并行 scan、硬件布局和更完整的初始化策略。

> 实验边界：CPU、离线、单线程、短合成序列；受控记忆只能验证递推、梯度与接口，不能代表长上下文建模能力。

## 1. 状态空间与张量合同

对每个 inner channel d 和 state 维 n，连续参数 A[d,n] 被约束为负数。离散更新为：

h_t = exp(delta_t A) · h_(t-1) + delta_t B_t u_t

y_t = sum_n(C_t h_t) + D u_t

u、delta 为 [B,T,D_inner]，B、C 为 [B,T,N]，state 为 [B,D_inner,N]。delta、B、C 都由当前输入生成，所以模型能选择性地保留或忽略信息。显式 scan 时间复杂度为 O(B·T·D_inner·N)，推理缓存与序列长度无关。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED44 = 4407  # 计算并保存当前步骤的中间状态。
random.seed(SEED44)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED44)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE44 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

VOCAB44 = ["<pad>", "<bos>", "甲", "乙", "丙", "丁", "戊", "己"]  # 计算并保存当前步骤的中间状态。
PAD44, BOS44 = 0, 1  # 计算并保存当前步骤的中间状态。

assert DEVICE44.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len(VOCAB44) == len(set(VOCAB44))  # 用受控断言验证关键不变量。
assert VOCAB44[PAD44] == "<pad>" and VOCAB44[BOS44] == "<bos>"  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE44), "vocab": len(VOCAB44)})  # 执行当前语句以推进本节示例。

## 2. Selective scan：先用标量手算锁定离散公式

A 使用 -exp(clamp(A_log))，天然为负；delta 使用 softplus 后再限制到安全范围，保证 exp(delta·A) 位于 (0,1]。这不是唯一离散化，官方 selective SSM 对参数化、初始化和 kernel 有更多细节。

valid=False 时输出为零并清空 state；reset=True 在处理当前 token 前清空 state。这样可以安全处理 padding 后重新开始的新片段，而不会把上一段历史泄漏进来。

In [ ]:
class SelectiveScan44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def forward(self, u, delta, b_input, c_input, a, direct, valid_mask, reset_mask, state=None):  # 定义本节可复用的核心函数。
        if u.ndim != 3 or delta.shape != u.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("u/delta 必须是同形 [B,T,D]")  # 遇到非法合同立即显式失败。
        batch, length, inner_dim = u.shape  # 计算并保存当前步骤的中间状态。
        if b_input.ndim != 3 or c_input.shape != b_input.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("B/C 必须是同形 [B,T,N]")  # 遇到非法合同立即显式失败。
        if b_input.shape[:2] != (batch, length):  # 按当前条件选择后续控制路径。
            raise ValueError("B/C 的 batch/time 不匹配")  # 遇到非法合同立即显式失败。
        state_dim = b_input.shape[-1]  # 计算并保存当前步骤的中间状态。
        if a.shape != (inner_dim, state_dim) or direct.shape != (inner_dim,):  # 按当前条件选择后续控制路径。
            raise ValueError("A/D 形状错误")  # 遇到非法合同立即显式失败。
        if valid_mask.shape != (batch, length) or valid_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("valid_mask 合同错误")  # 遇到非法合同立即显式失败。
        if reset_mask.shape != (batch, length) or reset_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("reset_mask 合同错误")  # 遇到非法合同立即显式失败。
        if not all(torch.isfinite(x).all() for x in [u, delta, b_input, c_input, a, direct]):  # 按当前条件选择后续控制路径。
            raise ValueError("scan 输入必须有限")  # 遇到非法合同立即显式失败。
        if state is None:  # 按当前条件选择后续控制路径。
            state = u.new_zeros(batch, inner_dim, state_dim)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            if state.shape != (batch, inner_dim, state_dim):  # 按当前条件选择后续控制路径。
                raise ValueError("初始 state 形状错误")  # 遇到非法合同立即显式失败。
            if state.dtype != u.dtype or state.device != u.device:  # 按当前条件选择后续控制路径。
                raise ValueError("初始 state 的 dtype/device 必须与 u 一致")  # 遇到非法合同立即显式失败。
            if not torch.isfinite(state).all():  # 按当前条件选择后续控制路径。
                raise ValueError("初始 state 必须全部有限")  # 遇到非法合同立即显式失败。

        outputs = []  # 计算并保存当前步骤的中间状态。
        for position in range(length):  # 遍历输入元素以累积或检查结果。
            reset = reset_mask[:, position, None, None]  # 计算并保存当前步骤的中间状态。
            state = torch.where(reset, torch.zeros_like(state), state)  # 计算并保存当前步骤的中间状态。
            dt = delta[:, position, :, None]  # 计算并保存当前步骤的中间状态。
            decay = torch.exp(dt * a[None])  # 计算并保存当前步骤的中间状态。
            candidate = (  # 计算并保存当前步骤的中间状态。
                decay * state  # 执行当前语句以推进本节示例。
                + dt * b_input[:, position, None, :] * u[:, position, :, None]  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            valid = valid_mask[:, position, None, None]  # 计算并保存当前步骤的中间状态。
            state = torch.where(valid, candidate, torch.zeros_like(state))  # 计算并保存当前步骤的中间状态。
            y = (  # 计算并保存当前步骤的中间状态。
                (state * c_input[:, position, None, :]).sum(-1)  # 执行当前语句以推进本节示例。
                + direct[None] * u[:, position]  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            outputs.append(y * valid_mask[:, position, None])  # 执行当前语句以推进本节示例。
        return torch.stack(outputs, dim=1), state  # 返回当前分支计算出的结果。


scan44 = SelectiveScan44()  # 计算并保存当前步骤的中间状态。
u_oracle44 = torch.tensor([[[1.0], [2.0]]])  # 计算并保存当前步骤的中间状态。
delta_oracle44 = torch.full_like(u_oracle44, math.log(2.0))  # 计算并保存当前步骤的中间状态。
b_oracle44 = torch.full((1, 2, 1), 1.0 / math.log(2.0))  # 计算并保存当前步骤的中间状态。
c_oracle44 = torch.ones(1, 2, 1)  # 计算并保存当前步骤的中间状态。
a_oracle44 = torch.tensor([[-1.0]])  # 计算并保存当前步骤的中间状态。
d_oracle44 = torch.zeros(1)  # 计算并保存当前步骤的中间状态。
valid_oracle44 = torch.ones(1, 2, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
reset_oracle44 = torch.tensor([[True, False]])  # 计算并保存当前步骤的中间状态。
y_oracle44, state_oracle44 = scan44(  # 计算并保存当前步骤的中间状态。
    u_oracle44, delta_oracle44, b_oracle44, c_oracle44,  # 执行当前语句以推进本节示例。
    a_oracle44, d_oracle44, valid_oracle44, reset_oracle44,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert torch.allclose(y_oracle44.flatten(), torch.tensor([1.0, 2.5]), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(state_oracle44.flatten(), torch.tensor([2.5]), atol=1e-6)  # 用受控断言验证关键不变量。

reset_second44 = torch.tensor([[True, True]])  # 计算并保存当前步骤的中间状态。
y_reset44, state_reset44 = scan44(  # 计算并保存当前步骤的中间状态。
    u_oracle44, delta_oracle44, b_oracle44, c_oracle44,  # 执行当前语句以推进本节示例。
    a_oracle44, d_oracle44, valid_oracle44, reset_second44,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert torch.allclose(y_reset44.flatten(), torch.tensor([1.0, 2.0]), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(state_reset44.flatten(), torch.tensor([2.0]), atol=1e-6)  # 用受控断言验证关键不变量。

invalid_middle44 = torch.tensor([[True, False]])  # 计算并保存当前步骤的中间状态。
y_invalid44, state_invalid44 = scan44(  # 计算并保存当前步骤的中间状态。
    u_oracle44, delta_oracle44, b_oracle44, c_oracle44,  # 执行当前语句以推进本节示例。
    a_oracle44, d_oracle44, invalid_middle44, reset_oracle44,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert y_invalid44[0, 1, 0].item() == 0.0  # 用受控断言验证关键不变量。
assert state_invalid44.abs().max().item() == 0.0  # 用受控断言验证关键不变量。

for bad_state44 in [  # 遍历输入元素以累积或检查结果。
    torch.zeros(1, 1, 2),  # 执行当前语句以推进本节示例。
    torch.zeros(1, 1, 1, dtype=torch.float64),  # 计算并保存当前步骤的中间状态。
    torch.full((1, 1, 1), float("nan")),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        scan44(  # 执行当前语句以推进本节示例。
            u_oracle44[:, :1], delta_oracle44[:, :1], b_oracle44[:, :1], c_oracle44[:, :1],  # 执行当前语句以推进本节示例。
            a_oracle44, d_oracle44, valid_oracle44[:, :1], reset_oracle44[:, :1], bad_state44,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        raise AssertionError("非法外部 SSM state 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。


for bad_state44 in [  # 遍历输入元素以累积或检查结果。
    torch.zeros(1, 1, 2),  # 执行当前语句以推进本节示例。
    torch.zeros(1, 1, 1, dtype=torch.float64),  # 计算并保存当前步骤的中间状态。
    torch.full((1, 1, 1), float("nan")),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        scan44(  # 执行当前语句以推进本节示例。
            u_oracle44[:, :1], delta_oracle44[:, :1], b_oracle44[:, :1], c_oracle44[:, :1],  # 执行当前语句以推进本节示例。
            a_oracle44, d_oracle44, valid_oracle44[:, :1], reset_oracle44[:, :1], bad_state44,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        raise AssertionError("非法外部 SSM state 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。


for bad_state44 in [  # 遍历输入元素以累积或检查结果。
    torch.zeros(1, 1, 2),  # 执行当前语句以推进本节示例。
    torch.zeros(1, 1, 1, dtype=torch.float64),  # 计算并保存当前步骤的中间状态。
    torch.full((1, 1, 1), float("nan")),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        scan44(  # 执行当前语句以推进本节示例。
            u_oracle44[:, :1], delta_oracle44[:, :1], b_oracle44[:, :1], c_oracle44[:, :1],  # 执行当前语句以推进本节示例。
            a_oracle44, d_oracle44, valid_oracle44[:, :1], reset_oracle44[:, :1], bad_state44,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        raise AssertionError("非法外部 SSM state 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。


## 3. Depthwise causal convolution 与固定大小缓存

每个 inner channel 使用独立的一维 kernel，只看当前与左侧 K-1 个输入。full forward 可写成左侧补零后的 grouped conv1d；step forward 只需保存 [B,K-1,D_inner] 环形窗口。

教学实现的通用 forward 用显式 step 支持中途 reset，另提供 vectorized_no_reset 作为独立 oracle，确保窗口次序、权重翻转和 bias 没有写错。

In [ ]:
class CausalDepthwiseConv44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, channels, kernel_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if kernel_size < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("kernel_size 必须为正")  # 遇到非法合同立即显式失败。
        self.channels = channels  # 计算并保存当前步骤的中间状态。
        self.kernel_size = kernel_size  # 计算并保存当前步骤的中间状态。
        self.weight = nn.Parameter(torch.empty(channels, kernel_size))  # 计算并保存当前步骤的中间状态。
        self.bias = nn.Parameter(torch.zeros(channels))  # 计算并保存当前步骤的中间状态。
        nn.init.normal_(self.weight, std=0.1)  # 计算并保存当前步骤的中间状态。

    def init_state(self, batch, device, dtype):  # 定义本节可复用的核心函数。
        return torch.zeros(batch, self.kernel_size - 1, self.channels, device=device, dtype=dtype)  # 返回当前分支计算出的结果。

    def step(self, x, state, valid, reset):  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.channels:  # 按当前条件选择后续控制路径。
            raise ValueError("conv step x 形状错误")  # 遇到非法合同立即显式失败。
        expected_state = (x.shape[0], self.kernel_size - 1, self.channels)  # 计算并保存当前步骤的中间状态。
        if state.shape != expected_state:  # 按当前条件选择后续控制路径。
            raise ValueError("conv cache 形状错误")  # 遇到非法合同立即显式失败。
        if state.dtype != x.dtype or state.device != x.device:  # 按当前条件选择后续控制路径。
            raise ValueError("conv cache 的 dtype/device 必须与 x 一致")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(x).all() or not torch.isfinite(state).all():  # 按当前条件选择后续控制路径。
            raise ValueError("conv 输入与 cache 必须全部有限")  # 遇到非法合同立即显式失败。
        if valid.shape != (x.shape[0],) or reset.shape != valid.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("conv step mask 形状错误")  # 遇到非法合同立即显式失败。
        if valid.dtype != torch.bool or reset.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("conv step mask 必须是 bool")  # 遇到非法合同立即显式失败。
        state = torch.where(reset[:, None, None], torch.zeros_like(state), state)  # 计算并保存当前步骤的中间状态。
        window = torch.cat([state, x[:, None]], dim=1)  # 计算并保存当前步骤的中间状态。
        y = (window.transpose(1, 2) * self.weight[None]).sum(-1) + self.bias  # 计算并保存当前步骤的中间状态。
        if self.kernel_size > 1:  # 按当前条件选择后续控制路径。
            next_state = window[:, 1:]  # 计算并保存当前步骤的中间状态。
            next_state = torch.where(valid[:, None, None], next_state, torch.zeros_like(next_state))  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            next_state = state  # 计算并保存当前步骤的中间状态。
        return y * valid[:, None], next_state  # 返回当前分支计算出的结果。

    def forward(self, x, valid_mask, reset_mask, state=None):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or x.shape[-1] != self.channels or x.shape[1] == 0:  # 按当前条件选择后续控制路径。
            raise ValueError("conv 输入必须是非空 [B,T,C]")  # 遇到非法合同立即显式失败。
        if valid_mask.shape != x.shape[:2] or reset_mask.shape != x.shape[:2]:  # 按当前条件选择后续控制路径。
            raise ValueError("conv mask 形状错误")  # 遇到非法合同立即显式失败。
        if valid_mask.dtype != torch.bool or reset_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("conv mask 必须是 bool")  # 遇到非法合同立即显式失败。
        if state is None:  # 按当前条件选择后续控制路径。
            state = self.init_state(x.shape[0], x.device, x.dtype)  # 计算并保存当前步骤的中间状态。
        outputs = []  # 计算并保存当前步骤的中间状态。
        for position in range(x.shape[1]):  # 遍历输入元素以累积或检查结果。
            y, state = self.step(  # 计算并保存当前步骤的中间状态。
                x[:, position], state,  # 执行当前语句以推进本节示例。
                valid_mask[:, position], reset_mask[:, position],  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            outputs.append(y)  # 执行当前语句以推进本节示例。
        return torch.stack(outputs, dim=1), state  # 返回当前分支计算出的结果。

    def vectorized_no_reset(self, x):  # 定义本节可复用的核心函数。
        padded = F.pad(x.transpose(1, 2), (self.kernel_size - 1, 0))  # 计算并保存当前步骤的中间状态。
        return F.conv1d(  # 返回当前分支计算出的结果。
            padded, self.weight[:, None, :], self.bias, groups=self.channels  # 计算并保存当前步骤的中间状态。
        ).transpose(1, 2)  # 执行当前语句以推进本节示例。


conv_probe44 = CausalDepthwiseConv44(2, 3)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    conv_probe44.weight.copy_(torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]]))  # 执行当前语句以推进本节示例。
    conv_probe44.bias.copy_(torch.tensor([0.25, -0.5]))  # 执行当前语句以推进本节示例。
conv_x44 = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]]])  # 计算并保存当前步骤的中间状态。
conv_valid44 = torch.ones(1, 4, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
conv_reset44 = torch.tensor([[True, False, False, False]])  # 计算并保存当前步骤的中间状态。
conv_loop44, conv_state44 = conv_probe44(conv_x44, conv_valid44, conv_reset44)  # 计算并保存当前步骤的中间状态。
conv_vector44 = conv_probe44.vectorized_no_reset(conv_x44)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(conv_loop44, conv_vector44, atol=1e-7)  # 用受控断言验证关键不变量。
assert conv_state44.shape == (1, 2, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(conv_state44[0, :, 0], torch.tensor([5.0, 7.0]))  # 用受控断言验证关键不变量。

future_changed44 = conv_x44.clone()  # 计算并保存当前步骤的中间状态。
future_changed44[:, 3] += 100.0  # 计算并保存当前步骤的中间状态。
future_output44, _ = conv_probe44(future_changed44, conv_valid44, conv_reset44)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(conv_loop44[:, :3], future_output44[:, :3], atol=1e-7)  # 用受控断言验证关键不变量。

## 4. Mamba 风格 mixer：input-dependent delta/B/C 与 gate

input projection 一分为二：x 分支经过 causal conv 与 SiLU 后产生 scan 输入；z 分支提供 SiLU gate。parameter projection 从每个时间步的 conv 特征生成 delta_raw[d]、B[n]、C[n]。

A=-exp(clamp(A_log,-8,4)) 保证负数；delta=clamp(softplus(delta_raw+dt_bias),1e-4,1) 防止极端指数。clamp 是本教学实现的数值护栏，生产模型需用与训练 checkpoint 一致的精确参数化。

In [ ]:
class RMSNorm44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, eps=1e-6):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.weight = nn.Parameter(torch.ones(dim))  # 计算并保存当前步骤的中间状态。
        self.eps = eps  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.weight  # 返回当前分支计算出的结果。


class MambaMixer44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, inner_dim, state_dim, kernel_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.dim, self.inner_dim, self.state_dim = dim, inner_dim, state_dim  # 计算并保存当前步骤的中间状态。
        self.in_proj = nn.Linear(dim, 2 * inner_dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.conv = CausalDepthwiseConv44(inner_dim, kernel_size)  # 计算并保存当前步骤的中间状态。
        self.parameter_proj = nn.Linear(inner_dim, inner_dim + 2 * state_dim)  # 计算并保存当前步骤的中间状态。
        self.dt_bias = nn.Parameter(torch.full((inner_dim,), -2.0))  # 计算并保存当前步骤的中间状态。
        self.a_log = nn.Parameter(torch.zeros(inner_dim, state_dim))  # 计算并保存当前步骤的中间状态。
        self.direct = nn.Parameter(torch.ones(inner_dim))  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(inner_dim, dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.scan = SelectiveScan44()  # 计算并保存当前步骤的中间状态。

    def stable_parameters(self, conv_features):  # 定义本节可复用的核心函数。
        packed = self.parameter_proj(conv_features)  # 计算并保存当前步骤的中间状态。
        delta_raw, b_input, c_input = torch.split(  # 计算并保存当前步骤的中间状态。
            packed, [self.inner_dim, self.state_dim, self.state_dim], dim=-1  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        delta = F.softplus(delta_raw + self.dt_bias).clamp(1e-4, 1.0)  # 计算并保存当前步骤的中间状态。
        a = -torch.exp(self.a_log.clamp(-8.0, 4.0))  # 计算并保存当前步骤的中间状态。
        return delta, b_input, c_input, a  # 返回当前分支计算出的结果。

    def forward(self, x, valid_mask, reset_mask):  # 定义本节可复用的核心函数。
        projected, gate = self.in_proj(x).chunk(2, dim=-1)  # 计算并保存当前步骤的中间状态。
        convolved, conv_state = self.conv(projected, valid_mask, reset_mask)  # 计算并保存当前步骤的中间状态。
        conv_features = F.silu(convolved)  # 计算并保存当前步骤的中间状态。
        delta, b_input, c_input, a = self.stable_parameters(conv_features)  # 计算并保存当前步骤的中间状态。
        scanned, ssm_state = self.scan(  # 计算并保存当前步骤的中间状态。
            conv_features, delta, b_input, c_input, a, self.direct,  # 执行当前语句以推进本节示例。
            valid_mask, reset_mask,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        output = self.out_proj(scanned * F.silu(gate)) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return output, (conv_state, ssm_state)  # 返回当前分支计算出的结果。

    def init_cache(self, batch, device, dtype):  # 定义本节可复用的核心函数。
        return (  # 返回当前分支计算出的结果。
            self.conv.init_state(batch, device, dtype),  # 执行当前语句以推进本节示例。
            torch.zeros(batch, self.inner_dim, self.state_dim, device=device, dtype=dtype),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。

    def step(self, x, valid, reset, cache):  # 定义本节可复用的核心函数。
        if not isinstance(cache, (tuple, list)) or len(cache) != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("mixer cache 必须包含 conv/SSM 两部分")  # 遇到非法合同立即显式失败。
        projected, gate = self.in_proj(x).chunk(2, dim=-1)  # 计算并保存当前步骤的中间状态。
        conv_features, conv_state = self.conv.step(projected, cache[0], valid, reset)  # 计算并保存当前步骤的中间状态。
        conv_features = F.silu(conv_features)  # 计算并保存当前步骤的中间状态。
        delta, b_input, c_input, a = self.stable_parameters(conv_features[:, None])  # 计算并保存当前步骤的中间状态。
        scanned, ssm_state = self.scan(  # 计算并保存当前步骤的中间状态。
            conv_features[:, None], delta, b_input, c_input, a, self.direct,  # 执行当前语句以推进本节示例。
            valid[:, None], reset[:, None], cache[1],  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        output = self.out_proj(scanned[:, 0] * F.silu(gate)) * valid[:, None]  # 计算并保存当前步骤的中间状态。
        return output, (conv_state, ssm_state)  # 返回当前分支计算出的结果。


mixer_probe44 = MambaMixer44(8, 12, 3, 3)  # 计算并保存当前步骤的中间状态。
mixer_input44 = torch.randn(2, 5, 8)  # 计算并保存当前步骤的中间状态。
mixer_mask44 = torch.tensor([[True] * 5, [True, True, True, False, False]])  # 计算并保存当前步骤的中间状态。
mixer_reset44 = torch.zeros_like(mixer_mask44)  # 计算并保存当前步骤的中间状态。
mixer_reset44[:, 0] = True  # 计算并保存当前步骤的中间状态。
mixer_output44, mixer_cache44 = mixer_probe44(mixer_input44, mixer_mask44, mixer_reset44)  # 计算并保存当前步骤的中间状态。
assert mixer_output44.shape == mixer_input44.shape  # 用受控断言验证关键不变量。
assert mixer_cache44[0].shape == (2, 2, 12)  # 用受控断言验证关键不变量。
assert mixer_cache44[1].shape == (2, 12, 3)  # 用受控断言验证关键不变量。
assert mixer_output44[1, 3:].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
delta_probe44, _, _, a_probe44 = mixer_probe44.stable_parameters(torch.full((1, 1, 12), 1e6))  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(delta_probe44).all() and torch.isfinite(a_probe44).all()  # 用受控断言验证关键不变量。
assert bool((delta_probe44 >= 1e-4).all() and (delta_probe44 <= 1.0).all())  # 用受控断言验证关键不变量。
assert bool((a_probe44 < 0).all())  # 用受控断言验证关键不变量。

## 5. Residual block、完整 LM 与 step cache

block 使用 pre-norm 后调用 mixer 并做 residual。完整模型没有绝对位置 embedding，时序信息来自 causal conv 与递推 state。step 接口为每层保存 conv window 和 SSM state；因此生成第 t 个 token 不需要重算 0..t-1。

full、逐 token 与任意 chunk 切分必须给出相同 logits。这条等价测试比单纯检查 shape 更容易发现 cache 更新顺序、reset 时机或卷积窗口的 off-by-one。

这里的 `forward_chunk` 是实际公开接口：接收上一 chunk 的两类逐层 cache，并返回当前 logits 和下一 cache；不是把 step 循环仅仅换个分组名字。`forward(..., return_cache=True)` 走完整序列路径，因此可以同时比较 full、step、非规则 chunks 的 logits 与最终 conv/SSM state。

服务层不能让调用方裸持有可跨请求复用的 cache。`begin_request44` 将 cache 绑定非空 request ID，`request_step44` 拒绝 ID 不一致的 stale cache，并要求每个请求的首个有效 token 明确 reset；请求内部 invalid step 仍按既有语义清空状态。


In [ ]:
class MambaBlock44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, inner_dim, state_dim, kernel_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm = RMSNorm44(dim)  # 计算并保存当前步骤的中间状态。
        self.mixer = MambaMixer44(dim, inner_dim, state_dim, kernel_size)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, valid_mask, reset_mask):  # 定义本节可复用的核心函数。
        update, cache = self.mixer(self.norm(x), valid_mask, reset_mask)  # 计算并保存当前步骤的中间状态。
        return (x + update) * valid_mask.unsqueeze(-1), cache  # 返回当前分支计算出的结果。

    def init_cache(self, batch, device, dtype):  # 定义本节可复用的核心函数。
        return self.mixer.init_cache(batch, device, dtype)  # 返回当前分支计算出的结果。

    def step(self, x, valid, reset, cache):  # 定义本节可复用的核心函数。
        update, cache = self.mixer.step(self.norm(x), valid, reset, cache)  # 计算并保存当前步骤的中间状态。
        return (x + update) * valid[:, None], cache  # 返回当前分支计算出的结果。


class TinyMambaLM44(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, dim=16, inner_dim=24, state_dim=4, kernel_size=3, layers=1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = {  # 计算并保存当前步骤的中间状态。
            "vocab_size": vocab_size, "dim": dim, "inner_dim": inner_dim,  # 执行当前语句以推进本节示例。
            "state_dim": state_dim, "kernel_size": kernel_size, "layers": layers,  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=PAD44)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList(  # 计算并保存当前步骤的中间状态。
            [MambaBlock44(dim, inner_dim, state_dim, kernel_size) for _ in range(layers)]  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.final_norm = RMSNorm44(dim)  # 计算并保存当前步骤的中间状态。
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)  # 计算并保存当前步骤的中间状态。
        self.lm_head.weight = self.embedding.weight  # 计算并保存当前步骤的中间状态。

    def _validate(self, token_ids, valid_mask):  # 定义本节可复用的核心函数。
        if token_ids.dtype != torch.long or token_ids.ndim != 2 or token_ids.shape[1] == 0:  # 按当前条件选择后续控制路径。
            raise ValueError("token_ids 必须是非空二维 long")  # 遇到非法合同立即显式失败。
        if valid_mask.shape != token_ids.shape or valid_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("valid_mask 合同错误")  # 遇到非法合同立即显式失败。
        seen_padding = (~valid_mask).cumsum(1) > 0  # 计算并保存当前步骤的中间状态。
        if bool((valid_mask & seen_padding).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("batch forward 只接受右 padding")  # 遇到非法合同立即显式失败。

    def forward(self, token_ids, valid_mask, return_cache=False):  # 定义本节可复用的核心函数。
        self._validate(token_ids, valid_mask)  # 执行当前语句以推进本节示例。
        reset_mask = torch.zeros_like(valid_mask)  # 计算并保存当前步骤的中间状态。
        reset_mask[:, 0] = True  # 计算并保存当前步骤的中间状态。
        hidden = self.embedding(token_ids) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        final_cache = []  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            hidden, layer_cache = block(hidden, valid_mask, reset_mask)  # 计算并保存当前步骤的中间状态。
            final_cache.append(layer_cache)  # 执行当前语句以推进本节示例。
        hidden = self.final_norm(hidden) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        logits = self.lm_head(hidden) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return (logits, final_cache) if return_cache else logits  # 返回当前分支计算出的结果。

    def init_cache(self, batch, device=None):  # 定义本节可复用的核心函数。
        if batch <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("cache batch 必须为正")  # 遇到非法合同立即显式失败。
        device = device or self.embedding.weight.device  # 计算并保存当前步骤的中间状态。
        dtype = self.embedding.weight.dtype  # 计算并保存当前步骤的中间状态。
        return [block.init_cache(batch, device, dtype) for block in self.blocks]  # 返回当前分支计算出的结果。

    def step(self, token_ids, valid, reset, cache):  # 定义本节可复用的核心函数。
        if token_ids.dtype != torch.long or token_ids.ndim != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("step token_ids 必须是 [B] long")  # 遇到非法合同立即显式失败。
        if valid.shape != token_ids.shape or reset.shape != token_ids.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("step mask 形状错误")  # 遇到非法合同立即显式失败。
        if valid.dtype != torch.bool or reset.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("step mask 必须是 bool")  # 遇到非法合同立即显式失败。
        if not isinstance(cache, list) or len(cache) != len(self.blocks):  # 按当前条件选择后续控制路径。
            raise ValueError("cache 层数错误")  # 遇到非法合同立即显式失败。
        hidden = self.embedding(token_ids) * valid[:, None]  # 计算并保存当前步骤的中间状态。
        next_cache = []  # 计算并保存当前步骤的中间状态。
        for block, layer_cache in zip(self.blocks, cache):  # 遍历输入元素以累积或检查结果。
            hidden, layer_cache = block.step(hidden, valid, reset, layer_cache)  # 计算并保存当前步骤的中间状态。
            next_cache.append(layer_cache)  # 执行当前语句以推进本节示例。
        hidden = self.final_norm(hidden) * valid[:, None]  # 计算并保存当前步骤的中间状态。
        return self.lm_head(hidden) * valid[:, None], next_cache  # 返回当前分支计算出的结果。

    def forward_chunk(self, token_ids, valid_mask, reset_mask, cache):  # 定义本节可复用的核心函数。
        if token_ids.dtype != torch.long or token_ids.ndim != 2 or token_ids.shape[1] == 0:  # 按当前条件选择后续控制路径。
            raise ValueError("chunk token_ids 必须是非空二维 long")  # 遇到非法合同立即显式失败。
        if valid_mask.shape != token_ids.shape or reset_mask.shape != token_ids.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("chunk mask 形状错误")  # 遇到非法合同立即显式失败。
        if valid_mask.dtype != torch.bool or reset_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("chunk mask 必须是 bool")  # 遇到非法合同立即显式失败。
        outputs = []  # 计算并保存当前步骤的中间状态。
        for position in range(token_ids.shape[1]):  # 遍历输入元素以累积或检查结果。
            logits, cache = self.step(  # 计算并保存当前步骤的中间状态。
                token_ids[:, position], valid_mask[:, position], reset_mask[:, position], cache  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
            outputs.append(logits)  # 执行当前语句以推进本节示例。
        return torch.stack(outputs, dim=1), cache  # 返回当前分支计算出的结果。


class RequestScopedCache44:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, request_id, batch_size, layers):  # 定义本节可复用的核心函数。
        if not isinstance(request_id, str) or not request_id:  # 按当前条件选择后续控制路径。
            raise ValueError("request_id 必须是非空字符串")  # 遇到非法合同立即显式失败。
        self.request_id = request_id  # 计算并保存当前步骤的中间状态。
        self.batch_size = int(batch_size)  # 计算并保存当前步骤的中间状态。
        self.layers = layers  # 计算并保存当前步骤的中间状态。
        self.seen_valid = torch.zeros(batch_size, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。


def begin_request44(model, request_id, batch_size):  # 定义本节可复用的核心函数。
    return RequestScopedCache44(request_id, batch_size, model.init_cache(batch_size))  # 返回当前分支计算出的结果。


def request_step44(model, request_id, token_ids, valid, reset, scoped_cache):  # 定义本节可复用的核心函数。
    if not isinstance(scoped_cache, RequestScopedCache44) or scoped_cache.request_id != request_id:  # 按当前条件选择后续控制路径。
        raise ValueError("cache 不属于当前 request，拒绝跨请求复用")  # 遇到非法合同立即显式失败。
    if token_ids.shape != (scoped_cache.batch_size,):  # 按当前条件选择后续控制路径。
        raise ValueError("request batch 与 cache 不匹配")  # 遇到非法合同立即显式失败。
    first_valid = valid & ~scoped_cache.seen_valid  # 计算并保存当前步骤的中间状态。
    if bool((first_valid & ~reset).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("每个 request 的首个有效 token 必须 reset")  # 遇到非法合同立即显式失败。
    logits, next_layers = model.step(token_ids, valid, reset, scoped_cache.layers)  # 计算并保存当前步骤的中间状态。
    scoped_cache.layers = next_layers  # 计算并保存当前步骤的中间状态。
    scoped_cache.seen_valid |= valid  # 计算并保存当前步骤的中间状态。
    return logits, scoped_cache  # 返回当前分支计算出的结果。


torch.manual_seed(SEED44)  # 执行当前语句以推进本节示例。
model44 = TinyMambaLM44(len(VOCAB44))  # 计算并保存当前步骤的中间状态。
ids_probe44 = torch.tensor([[BOS44, 2, 3, 4, 5], [BOS44, 6, 7, PAD44, PAD44]])  # 计算并保存当前步骤的中间状态。
mask_probe44 = ids_probe44.ne(PAD44)  # 计算并保存当前步骤的中间状态。
full_logits44, full_cache44 = model44(ids_probe44, mask_probe44, return_cache=True)  # 计算并保存当前步骤的中间状态。

cache44 = model44.init_cache(ids_probe44.shape[0])  # 计算并保存当前步骤的中间状态。
step_logits44 = []  # 计算并保存当前步骤的中间状态。
for position in range(ids_probe44.shape[1]):  # 遍历输入元素以累积或检查结果。
    reset44 = torch.full((ids_probe44.shape[0],), position == 0, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    logits44, cache44 = model44.step(  # 计算并保存当前步骤的中间状态。
        ids_probe44[:, position], mask_probe44[:, position], reset44, cache44  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    step_logits44.append(logits44)  # 执行当前语句以推进本节示例。
step_logits44 = torch.stack(step_logits44, dim=1)  # 计算并保存当前步骤的中间状态。
assert full_logits44.shape == (2, 5, len(VOCAB44))  # 用受控断言验证关键不变量。
assert torch.allclose(full_logits44, step_logits44, atol=2e-6)  # 用受控断言验证关键不变量。
assert step_logits44[1, 3:].abs().max().item() == 0.0  # 用受控断言验证关键不变量。

cache_chunk44 = model44.init_cache(ids_probe44.shape[0])  # 计算并保存当前步骤的中间状态。
chunk_outputs44 = []  # 计算并保存当前步骤的中间状态。
for start, end in [(0, 2), (2, 3), (3, 5)]:  # 遍历输入元素以累积或检查结果。
    chunk_reset44 = torch.zeros_like(mask_probe44[:, start:end])  # 计算并保存当前步骤的中间状态。
    if start == 0:  # 按当前条件选择后续控制路径。
        chunk_reset44[:, 0] = True  # 计算并保存当前步骤的中间状态。
    chunk_logits44, cache_chunk44 = model44.forward_chunk(  # 计算并保存当前步骤的中间状态。
        ids_probe44[:, start:end], mask_probe44[:, start:end], chunk_reset44, cache_chunk44,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    chunk_outputs44.append(chunk_logits44)  # 执行当前语句以推进本节示例。
chunk_logits44 = torch.cat(chunk_outputs44, dim=1)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(chunk_logits44, full_logits44, atol=2e-6)  # 用受控断言验证关键不变量。

def assert_cache_close44(left, right):  # 定义本节可复用的核心函数。
    assert len(left) == len(right)  # 用受控断言验证关键不变量。
    for (left_conv, left_ssm), (right_conv, right_ssm) in zip(left, right):  # 遍历输入元素以累积或检查结果。
        assert torch.allclose(left_conv, right_conv, atol=2e-6)  # 用受控断言验证关键不变量。
        assert torch.allclose(left_ssm, right_ssm, atol=2e-6)  # 用受控断言验证关键不变量。

assert_cache_close44(full_cache44, cache44)  # 用受控断言验证关键不变量。
assert_cache_close44(full_cache44, cache_chunk44)  # 用受控断言验证关键不变量。
assert model44.lm_head.weight is model44.embedding.weight  # 用受控断言验证关键不变量。


## 6. 未来隔离、padding 后重置与梯度稳定性

因果模型的三个关键不变性是：改变未来 token 不影响前缀；右侧 padding 的 token 值不影响有效 logits；无效步清空两类 cache 后，下一个有效 token 与新序列首 token 等价。

数值测试还要覆盖极端输入、finite gradient 与非零状态参数梯度。仅观察训练 loss 无法定位递推公式是否正确。

In [ ]:
model44.eval()  # 执行当前语句以推进本节示例。
sequence_a44 = torch.tensor([[BOS44, 2, 3, 4, 5]])  # 计算并保存当前步骤的中间状态。
sequence_b44 = sequence_a44.clone()  # 计算并保存当前步骤的中间状态。
sequence_b44[0, 4] = 7  # 计算并保存当前步骤的中间状态。
visible44 = torch.ones_like(sequence_a44, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    causal_a44 = model44(sequence_a44, visible44)  # 计算并保存当前步骤的中间状态。
    causal_b44 = model44(sequence_b44, visible44)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(causal_a44[:, :4], causal_b44[:, :4], atol=2e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(causal_a44[:, 4], causal_b44[:, 4])  # 用受控断言验证关键不变量。

padded_a44 = torch.tensor([[BOS44, 2, 3, 4, 5, 6, 7]])  # 计算并保存当前步骤的中间状态。
padded_b44 = padded_a44.clone()  # 计算并保存当前步骤的中间状态。
padded_b44[0, 5:] = torch.tensor([2, 3])  # 计算并保存当前步骤的中间状态。
padded_mask44 = torch.tensor([[True, True, True, True, True, False, False]])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    output_a44 = model44(padded_a44, padded_mask44)  # 计算并保存当前步骤的中间状态。
    output_b44 = model44(padded_b44, padded_mask44)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(output_a44, output_b44, atol=2e-6)  # 用受控断言验证关键不变量。
assert output_a44[:, 5:].abs().max().item() == 0.0  # 用受控断言验证关键不变量。

cache_gap44 = model44.init_cache(1)  # 计算并保存当前步骤的中间状态。
gap_outputs44 = []  # 计算并保存当前步骤的中间状态。
for token, valid, reset in [  # 遍历输入元素以累积或检查结果。
    (2, True, True), (3, False, False), (4, True, False),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    output44, cache_gap44 = model44.step(  # 计算并保存当前步骤的中间状态。
        torch.tensor([token]), torch.tensor([valid]), torch.tensor([reset]), cache_gap44  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    gap_outputs44.append(output44)  # 执行当前语句以推进本节示例。
cache_fresh44 = model44.init_cache(1)  # 计算并保存当前步骤的中间状态。
fresh_output44, _ = model44.step(  # 计算并保存当前步骤的中间状态。
    torch.tensor([4]), torch.tensor([True]), torch.tensor([True]), cache_fresh44  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert gap_outputs44[1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
assert torch.allclose(gap_outputs44[2], fresh_output44, atol=2e-6)  # 用受控断言验证关键不变量。

# 外部 cache 的 shape/dtype/finite 必须逐层 fail closed。
for cache_mutator44 in ["nan-conv", "nan-ssm", "wrong-shape", "wrong-dtype"]:  # 遍历输入元素以累积或检查结果。
    bad_cache44 = model44.init_cache(1)  # 计算并保存当前步骤的中间状态。
    if cache_mutator44 == "nan-conv":  # 按当前条件选择后续控制路径。
        bad_cache44[0] = (torch.full_like(bad_cache44[0][0], float("nan")), bad_cache44[0][1])  # 计算并保存当前步骤的中间状态。
    elif cache_mutator44 == "nan-ssm":  # 按当前条件选择后续控制路径。
        bad_cache44[0] = (bad_cache44[0][0], torch.full_like(bad_cache44[0][1], float("nan")))  # 计算并保存当前步骤的中间状态。
    elif cache_mutator44 == "wrong-shape":  # 按当前条件选择后续控制路径。
        bad_cache44[0] = (torch.zeros(1, 1, 1), bad_cache44[0][1])  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        bad_cache44[0] = (bad_cache44[0][0].double(), bad_cache44[0][1])  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        model44.step(torch.tensor([2]), torch.tensor([True]), torch.tensor([True]), bad_cache44)  # 执行当前语句以推进本节示例。
        raise AssertionError("非法 layer cache 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

request_a44 = begin_request44(model44, "request-a", 1)  # 计算并保存当前步骤的中间状态。
_, request_a44 = request_step44(  # 计算并保存当前步骤的中间状态。
    model44, "request-a", torch.tensor([BOS44]), torch.tensor([True]), torch.tensor([True]), request_a44,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    request_step44(  # 执行当前语句以推进本节示例。
        model44, "request-b", torch.tensor([2]), torch.tensor([True]), torch.tensor([False]), request_a44,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("跨 request 复用 stale cache 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as request_error44:  # 捕获预期异常并验证失败分支。
    assert "跨请求" in str(request_error44) or "不属于" in str(request_error44)  # 用受控断言验证关键不变量。

request_c44 = begin_request44(model44, "request-c", 1)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    request_step44(  # 执行当前语句以推进本节示例。
        model44, "request-c", torch.tensor([2]), torch.tensor([True]), torch.tensor([False]), request_c44,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("request 首个有效 token 未 reset")  # 遇到非法合同立即显式失败。
except ValueError as reset_error44:  # 捕获预期异常并验证失败分支。
    assert "首个有效 token" in str(reset_error44)  # 用受控断言验证关键不变量。

model44.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
gradient_input44 = torch.tensor([[BOS44, 2, 3, 4]])  # 计算并保存当前步骤的中间状态。
gradient_mask44 = torch.ones_like(gradient_input44, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
gradient_loss44 = model44(gradient_input44, gradient_mask44).square().mean()  # 计算并保存当前步骤的中间状态。
gradient_loss44.backward()  # 执行当前语句以推进本节示例。
gradients44 = [p.grad for p in model44.parameters() if p.grad is not None]  # 计算并保存当前步骤的中间状态。
assert gradients44 and all(torch.isfinite(gradient).all() for gradient in gradients44)  # 用受控断言验证关键不变量。
assert model44.blocks[0].mixer.a_log.grad.abs().sum().item() > 0  # 用受控断言验证关键不变量。
assert model44.blocks[0].mixer.parameter_proj.weight.grad.abs().sum().item() > 0  # 用受控断言验证关键不变量。

## 7. 受控 next-token 训练

数据是六条固定循环序列，目标是预测下一个符号。训练集和验证集按记录 id 划分，完整内容稍后绑定进发布 manifest。四条训练序列都以 BOS 开始却有不同的首个目标，因此 token accuracy 存在不可约歧义；不能为了“满分”掩盖数据合同。loss 快速下降只说明 causal conv、scan、gate、residual 和 CE 串通，不表示能处理自然语言或十万 token 长上下文。

In [ ]:
RECORDS44 = [  # 计算并保存当前步骤的中间状态。
    {"id": "s0", "tokens": [1, 2, 3, 4, 5, 6]},  # 执行当前语句以推进本节示例。
    {"id": "s1", "tokens": [1, 3, 4, 5, 6, 7]},  # 执行当前语句以推进本节示例。
    {"id": "s2", "tokens": [1, 4, 5, 6, 7, 2]},  # 执行当前语句以推进本节示例。
    {"id": "s3", "tokens": [1, 5, 6, 7, 2, 3]},  # 执行当前语句以推进本节示例。
    {"id": "s4", "tokens": [1, 6, 7, 2, 3, 4]},  # 执行当前语句以推进本节示例。
    {"id": "s5", "tokens": [1, 7, 2, 3, 4, 5]},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
TRAIN_IDS44, VALID_IDS44 = ["s0", "s1", "s2", "s3"], ["s4", "s5"]  # 计算并保存当前步骤的中间状态。
train_records44 = [record for record in RECORDS44 if record["id"] in TRAIN_IDS44]  # 计算并保存当前步骤的中间状态。
train_inputs44 = torch.tensor([record["tokens"][:-1] for record in train_records44])  # 计算并保存当前步骤的中间状态。
train_targets44 = torch.tensor([record["tokens"][1:] for record in train_records44])  # 计算并保存当前步骤的中间状态。
train_mask44 = torch.ones_like(train_inputs44, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。

torch.manual_seed(SEED44)  # 执行当前语句以推进本节示例。
model44 = TinyMambaLM44(len(VOCAB44), dim=16, inner_dim=24, state_dim=4, kernel_size=3, layers=1)  # 计算并保存当前步骤的中间状态。
optimizer44 = torch.optim.Adam(model44.parameters(), lr=0.02)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss44 = F.cross_entropy(  # 计算并保存当前步骤的中间状态。
        model44(train_inputs44, train_mask44).reshape(-1, len(VOCAB44)),  # 执行当前语句以推进本节示例。
        train_targets44.reshape(-1),  # 执行当前语句以推进本节示例。
    ).item()  # 执行当前语句以推进本节示例。

history44 = []  # 计算并保存当前步骤的中间状态。
model44.train()  # 执行当前语句以推进本节示例。
for step in range(45):  # 遍历输入元素以累积或检查结果。
    optimizer44.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits44 = model44(train_inputs44, train_mask44)  # 计算并保存当前步骤的中间状态。
    loss44 = F.cross_entropy(logits44.reshape(-1, len(VOCAB44)), train_targets44.reshape(-1))  # 计算并保存当前步骤的中间状态。
    loss44.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model44.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer44.step()  # 执行当前语句以推进本节示例。
    if step in {0, 9, 29, 44}:  # 按当前条件选择后续控制路径。
        history44.append((step, float(loss44.detach())))  # 执行当前语句以推进本节示例。

model44.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    trained_logits44 = model44(train_inputs44, train_mask44)  # 计算并保存当前步骤的中间状态。
    final_loss44 = F.cross_entropy(  # 计算并保存当前步骤的中间状态。
        trained_logits44.reshape(-1, len(VOCAB44)), train_targets44.reshape(-1)  # 执行当前语句以推进本节示例。
    ).item()  # 执行当前语句以推进本节示例。
    train_accuracy44 = trained_logits44.argmax(-1).eq(train_targets44).float().mean().item()  # 计算并保存当前步骤的中间状态。

assert math.isfinite(initial_loss44) and math.isfinite(final_loss44)  # 用受控断言验证关键不变量。
assert final_loss44 < initial_loss44 * 0.25  # 用受控断言验证关键不变量。
# 四条训练序列的首个输入都是 BOS，却对应四个不同目标；
# 因而 token accuracy 存在不可约歧义，不能伪造“满分记忆”。
assert train_accuracy44 > 0.8  # 用受控断言验证关键不变量。
assert len(set(TRAIN_IDS44) & set(VALID_IDS44)) == 0  # 用受控断言验证关键不变量。
assert set(TRAIN_IDS44) | set(VALID_IDS44) == {record["id"] for record in RECORDS44}  # 用受控断言验证关键不变量。
print({  # 执行当前语句以推进本节示例。
    "initial_loss": round(initial_loss44, 4),  # 执行当前语句以推进本节示例。
    "final_loss": round(final_loss44, 4),  # 执行当前语句以推进本节示例。
    "train_accuracy": round(train_accuracy44, 3),  # 执行当前语句以推进本节示例。
    "trace": history44,  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。

## 8. 制品发布：绑定状态、数据、split 与递推 recipe

state digest 按排序后的参数 key 写入 key、dtype、shape、原始 bytes。manifest 完整保存词表、六条数据、互斥 split、padding/reset 语义、delta/A 数值护栏以及训练 recipe。加载时还会重新检查 split 和架构语义。

package 中的 internal hash 只能发现意外损坏，不能证明发布者身份；外部只读 publisher registry 的整体指纹才是信任锚。攻击者替换全部内容并重算 internal hash 仍会 fail closed。

In [ ]:
def canonical_json44(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")  # 返回当前分支计算出的结果。


def canonical_state_digest44(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(canonical_json44({  # 执行当前语句以推进本节示例。
            "key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)  # 执行当前语句以推进本节示例。
        }))  # 执行当前语句以推进本节示例。
        digest.update(tensor.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def package_fingerprint44(package):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    digest.update(canonical_json44(package["manifest"]))  # 执行当前语句以推进本节示例。
    digest.update(canonical_state_digest44(package["state"]).encode("ascii"))  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


manifest44 = {  # 计算并保存当前步骤的中间状态。
    "subject": "mamba-selective-ssm-demo@1",  # 执行当前语句以推进本节示例。
    "architecture": copy.deepcopy(model44.config),  # 执行当前语句以推进本节示例。
    "vocab": list(VOCAB44),  # 执行当前语句以推进本节示例。
    "dataset": copy.deepcopy(RECORDS44),  # 执行当前语句以推进本节示例。
    "split": {"train": list(TRAIN_IDS44), "validation": list(VALID_IDS44)},  # 执行当前语句以推进本节示例。
    "preprocess": {  # 执行当前语句以推进本节示例。
        "tokenizer": "frozen-token-id-v1", "padding": "right",  # 执行当前语句以推进本节示例。
        "invalid_step": "zero-output-and-reset-state",  # 执行当前语句以推进本节示例。
        "segment_reset": "before-current-token",  # 执行当前语句以推进本节示例。
        "external_cache_validation": "shape-dtype-device-finite",  # 执行当前语句以推进本节示例。
        "serving_cache": "request-id-bound-first-valid-must-reset",  # 执行当前语句以推进本节示例。
        "chunk_api": "forward_chunk-returns-next-cache",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "recipe": {  # 执行当前语句以推进本节示例。
        "seed": SEED44, "optimizer": "Adam", "learning_rate": 0.02,  # 执行当前语句以推进本节示例。
        "steps": 45, "gradient_clip": 1.0, "objective": "next-token-ce",  # 执行当前语句以推进本节示例。
        "a_parameterization": "-exp(clamp(a_log,-8,4))",  # 执行当前语句以推进本节示例。
        "delta_parameterization": "clamp(softplus(raw+dt_bias),1e-4,1)",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
state44 = {key: value.detach().cpu().clone() for key, value in model44.state_dict().items()}  # 计算并保存当前步骤的中间状态。
package44 = {  # 计算并保存当前步骤的中间状态。
    "manifest": manifest44,  # 执行当前语句以推进本节示例。
    "state": state44,  # 执行当前语句以推进本节示例。
    "internal": {  # 执行当前语句以推进本节示例。
        "manifest_digest": hashlib.sha256(canonical_json44(manifest44)).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_digest": canonical_state_digest44(state44),  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
subject44 = manifest44["subject"]  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY44 = MappingProxyType({subject44: package_fingerprint44(package44)})  # 计算并保存当前步骤的中间状态。


def load_published_mamba44(package, subject):  # 定义本节可复用的核心函数。
    if subject not in PUBLISHER_REGISTRY44:  # 按当前条件选择后续控制路径。
        raise ValueError("未知发布 subject")  # 遇到非法合同立即显式失败。
    if package_fingerprint44(package) != PUBLISHER_REGISTRY44[subject]:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry 指纹不匹配")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    if manifest["subject"] != subject:  # 按当前条件选择后续控制路径。
        raise ValueError("subject 不匹配")  # 遇到非法合同立即显式失败。
    if package["internal"]["manifest_digest"] != hashlib.sha256(canonical_json44(manifest)).hexdigest():  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if package["internal"]["state_digest"] != canonical_state_digest44(package["state"]):  # 按当前条件选择后续控制路径。
        raise ValueError("state 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if manifest["vocab"] != VOCAB44:  # 按当前条件选择后续控制路径。
        raise ValueError("词表不匹配")  # 遇到非法合同立即显式失败。
    train_ids = set(manifest["split"]["train"])  # 计算并保存当前步骤的中间状态。
    validation_ids = set(manifest["split"]["validation"])  # 计算并保存当前步骤的中间状态。
    data_ids = {record["id"] for record in manifest["dataset"]}  # 计算并保存当前步骤的中间状态。
    if train_ids & validation_ids or train_ids | validation_ids != data_ids:  # 按当前条件选择后续控制路径。
        raise ValueError("split 非互斥或未覆盖")  # 遇到非法合同立即显式失败。
    expected_preprocess44 = {  # 计算并保存当前步骤的中间状态。
        "tokenizer": "frozen-token-id-v1", "padding": "right",  # 执行当前语句以推进本节示例。
        "invalid_step": "zero-output-and-reset-state",  # 执行当前语句以推进本节示例。
        "segment_reset": "before-current-token",  # 执行当前语句以推进本节示例。
        "external_cache_validation": "shape-dtype-device-finite",  # 执行当前语句以推进本节示例。
        "serving_cache": "request-id-bound-first-valid-must-reset",  # 执行当前语句以推进本节示例。
        "chunk_api": "forward_chunk-returns-next-cache",  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    if manifest["preprocess"] != expected_preprocess44:  # 按当前条件选择后续控制路径。
        raise ValueError("state/cache 语义不匹配")  # 遇到非法合同立即显式失败。
    if manifest["recipe"]["a_parameterization"] != "-exp(clamp(a_log,-8,4))":  # 按当前条件选择后续控制路径。
        raise ValueError("A 参数化不匹配")  # 遇到非法合同立即显式失败。
    loaded = TinyMambaLM44(**manifest["architecture"])  # 计算并保存当前步骤的中间状态。
    loaded.load_state_dict(package["state"], strict=True)  # 计算并保存当前步骤的中间状态。
    loaded.eval()  # 执行当前语句以推进本节示例。
    return loaded  # 返回当前分支计算出的结果。


loaded44 = load_published_mamba44(package44, subject44)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.allclose(  # 用受控断言验证关键不变量。
        loaded44(train_inputs44, train_mask44),  # 执行当前语句以推进本节示例。
        model44(train_inputs44, train_mask44),  # 执行当前语句以推进本节示例。
        atol=1e-7,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
assert canonical_state_digest44(state44) == package44["internal"]["state_digest"]  # 用受控断言验证关键不变量。

forged44 = copy.deepcopy(package44)  # 计算并保存当前步骤的中间状态。
key44 = sorted(forged44["state"])[-1]  # 计算并保存当前步骤的中间状态。
forged44["state"][key44].view(-1)[0] += 0.5  # 计算并保存当前步骤的中间状态。
forged44["internal"]["state_digest"] = canonical_state_digest44(forged44["state"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_mamba44(forged44, subject44)  # 执行当前语句以推进本节示例。
    raise AssertionError("重算内部 state hash 的伪造未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error44:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error44)  # 用受控断言验证关键不变量。

replacement44 = copy.deepcopy(package44)  # 计算并保存当前步骤的中间状态。
replacement44["manifest"]["preprocess"]["invalid_step"] = "keep-state"  # 计算并保存当前步骤的中间状态。
replacement_key44 = sorted(replacement44["state"])[0]  # 计算并保存当前步骤的中间状态。
replacement44["state"][replacement_key44].view(-1)[-1] -= 0.25  # 计算并保存当前步骤的中间状态。
replacement44["internal"]["manifest_digest"] = hashlib.sha256(  # 计算并保存当前步骤的中间状态。
    canonical_json44(replacement44["manifest"])  # 执行当前语句以推进本节示例。
).hexdigest()  # 执行当前语句以推进本节示例。
replacement44["internal"]["state_digest"] = canonical_state_digest44(  # 计算并保存当前步骤的中间状态。
    replacement44["state"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_published_mamba44(replacement44, subject44)  # 执行当前语句以推进本节示例。
    raise AssertionError("整体替换并重算内部 hash 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error44:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error44)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY44[subject44] = "forged"  # 计算并保存当前步骤的中间状态。
    raise AssertionError("registry 不应可写")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 9. 失败模式、生产差距与资料

易错点包括：把 A 写成正数导致状态爆炸；漏乘 delta·B·u；B/C 错广播到 channel；因果卷积窗口反序；无效步只把输出归零却继续污染 cache；full 与 step 使用不同参数化；把 Python scan 的速度当作 Mamba 的系统优势。

生产实现还需要 fused selective scan、混合精度稳定性、长序列压力测试、state cache 生命周期、分布式训练、checkpoint 兼容、吞吐/显存基准、编译器与硬件回归。公开结果只有在官方 kernel、相同数据和相同评估协议下才可比较。

原始资料：

- [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752)
- [官方 state-spaces/mamba 实现](https://github.com/state-spaces/mamba)
- [S4: Efficiently Modeling Long Sequences with Structured State Spaces](https://arxiv.org/abs/2111.00396)